# SafeRoad AI — Fine-tune YOLO11n cho camera giao thông

**Chạy trên Google Colab với GPU** (Runtime → Change runtime type → T4 GPU).

## Vì sao phải fine-tune

Đo trực tiếp trên tập MVTI bằng trọng số COCO gốc:

| Cấu hình | recall@0.5 | Ghi chú |
|---|---:|---|
| YOLO11n COCO, imgsz 640, conf 0.25 | **0.19** | gần như không thấy gì |
| YOLO11n COCO, imgsz 1600, conf 0.10 | 0.31 | nhiều cảnh báo giả |
| YOLO11n COCO, cắt ô 2×3 (SAHI) | 0.46 | chậm hơn ~6 lần |

Nguyên nhân là **lệch miền (domain gap)**, không phải model yếu: COCO chủ yếu là
ảnh chụp ngang tầm mắt, còn camera giao thông đặt cao 10–18 m nhìn chếch xuống,
đối tượng nhỏ và bị nén phối cảnh. Trong một khung hình thử, YOLO11n chỉ phát
hiện đúng **một** vật thể và gán nhầm nhãn `train` cho một chiếc ô tô.

MVTI có sẵn **14.488 bounding box trên 2.441 ảnh** — quá đủ để đóng phần lớn
khoảng cách đó.

## Một lưu ý quan trọng về lớp đối tượng

MVTI chỉ chứa **car, truck/bus, bicycle**. Không có *người đi bộ* và *xe máy* —
hai lớp quan trọng bậc nhất với giao thông Việt Nam. Nếu fine-tune "thẳng" trên
5 lớp, model sẽ **quên** hai lớp đó (catastrophic forgetting) vì không hề gặp
mẫu nào trong lúc train.

Notebook này xử lý bằng cách trộn thêm ảnh COCO có chứa `person` và `motorcycle`
vào tập train (phần 4). Nếu bỏ qua bước đó, hãy ghi rõ trong báo cáo rằng model
fine-tune chỉ dùng được cho 3 lớp.

## 1. Cài đặt & kiểm tra GPU

In [ ]:
!nvidia-smi
!pip install -q ultralytics

import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('CUDA khả dụng:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Hãy bật GPU: Runtime → Change runtime type → T4 GPU'

## 2. Tải dataset lên

Trên **máy của bạn**, chạy trước:

```bash
python scripts/export_yolo_dataset.py --root data/raw/mvti --view Infrastructure --out data/yolo_mvti
# rồi nén lại
python -c "import shutil; shutil.make_archive('yolo_mvti','zip','data/yolo_mvti')"
```

Sau đó tải `yolo_mvti.zip` lên Google Drive (khuyến nghị — nhanh và không mất khi
phiên Colab hết hạn) hoặc upload trực tiếp.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sửa đường dẫn cho khớp nơi bạn đặt file trên Drive
ZIP_PATH = '/content/drive/MyDrive/yolo_mvti.zip'

!unzip -q -o "{ZIP_PATH}" -d /content/yolo_mvti
!ls /content/yolo_mvti
!cat /content/yolo_mvti/dataset_stats.json

In [ ]:
# data.yaml chứa đường dẫn tuyệt đối của MÁY BẠN — phải sửa lại cho Colab.
from pathlib import Path

yaml_path = Path('/content/yolo_mvti/data.yaml')
text = yaml_path.read_text(encoding='utf-8')
text = '\n'.join(
    'path: /content/yolo_mvti' if line.startswith('path:') else line
    for line in text.splitlines()
)
yaml_path.write_text(text, encoding='utf-8')
print(text)

## 3. Đo baseline TRƯỚC khi train

Bước này bắt buộc: không có số liệu "trước" thì không chứng minh được fine-tune
đã cải thiện bao nhiêu. Đây chính là dòng đầu tiên của bảng so sánh trong báo cáo.

In [ ]:
from ultralytics import YOLO

baseline = YOLO('yolo11n.pt')

# COCO index → chỉ số lớp của SafeRoad, để so sánh công bằng trên cùng bộ nhãn.
# person=0→0, bicycle=1→1, motorcycle=3→2, car=2→3, bus=5/truck=7→4
baseline_metrics = baseline.val(
    data='/content/yolo_mvti/data.yaml',
    imgsz=960, conf=0.001, iou=0.6, split='val', plots=False,
)
print('BASELINE (COCO gốc, nhãn chưa khớp chỉ số → con số này chỉ mang tính tham khảo)')
print('  mAP50    :', round(float(baseline_metrics.box.map50), 4))
print('  mAP50-95 :', round(float(baseline_metrics.box.map), 4))

## 4. (Khuyến nghị) Trộn thêm mẫu person + motorcycle từ COCO

Bỏ qua ô này nếu bạn chấp nhận model chỉ nhận 3 lớp có trong MVTI.

Ô dưới tải một tập con COCO nhỏ chứa `person` và `motorcycle`, chuyển nhãn sang
đúng chỉ số của SafeRoad rồi trộn vào tập train. Nhờ đó model giữ được khả năng
nhận diện hai lớp dễ tổn thương nhất.

In [ ]:
import random, shutil
from pathlib import Path

# Bộ COCO128 nhỏ gọn có sẵn của Ultralytics — đủ để "neo" hai lớp còn thiếu.
!wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip -O /content/coco128.zip
!unzip -q -o /content/coco128.zip -d /content/

COCO_TO_SAFEROAD = {0: 0, 1: 1, 3: 2, 2: 3, 5: 4, 7: 4}  # person,bicycle,moto,car,bus,truck
src_img = Path('/content/coco128/images/train2017')
src_lbl = Path('/content/coco128/labels/train2017')
dst_img = Path('/content/yolo_mvti/images/train')
dst_lbl = Path('/content/yolo_mvti/labels/train')

added = 0
for lbl in sorted(src_lbl.glob('*.txt')):
    kept = []
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        cid = int(parts[0])
        if cid in COCO_TO_SAFEROAD:
            kept.append(' '.join([str(COCO_TO_SAFEROAD[cid])] + parts[1:]))
    if not kept:
        continue
    img = src_img / f'{lbl.stem}.jpg'
    if not img.exists():
        continue
    shutil.copy2(img, dst_img / f'coco_{img.name}')
    (dst_lbl / f'coco_{lbl.stem}.txt').write_text('\n'.join(kept))
    added += 1

print(f'Đã trộn thêm {added} ảnh COCO có person/motorcycle vào tập train')

## 5. Fine-tune

Các lựa chọn tham số và lý do:

* `imgsz=960` — đối tượng nhỏ, giữ độ phân giải cao hơn mặc định 640.
* `epochs=60`, `patience=15` — dừng sớm khi val không cải thiện, tránh overfit
  vào một giao lộ duy nhất.
* `freeze=10` — đóng băng 10 lớp backbone đầu. Với ~2.400 ảnh từ **một** hiện
  trường, train toàn bộ mạng rất dễ overfit; giữ lại đặc trưng thị giác tổng quát
  của COCO và chỉ dạy lại phần đầu ra.
* `degrees=0`, `perspective=0` — camera cố định nên **không** xoay/biến dạng ảnh;
  augment kiểu đó sẽ tạo ra góc nhìn không bao giờ tồn tại khi triển khai.
* `hsv_*`, `mosaic` giữ ở mức vừa phải để mô phỏng đổi ánh sáng và mật độ xe.

In [ ]:
model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/yolo_mvti/data.yaml',
    epochs=60,
    patience=15,
    imgsz=960,
    batch=16,
    freeze=10,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,
    # Camera cố định → không augment hình học mạnh
    degrees=0.0, perspective=0.0, shear=0.0, flipud=0.0, fliplr=0.5,
    scale=0.3, translate=0.1,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    mosaic=0.6, close_mosaic=10,
    project='/content/runs', name='saferoad_yolo11n',
    seed=42, deterministic=True, plots=True,
)

## 6. Đánh giá SAU khi train

In [ ]:
best = YOLO('/content/runs/saferoad_yolo11n/weights/best.pt')
m = best.val(data='/content/yolo_mvti/data.yaml', imgsz=960, split='val', plots=True)

print('SAU FINE-TUNE')
print('  mAP50    :', round(float(m.box.map50), 4), ' (mục tiêu poster ≥ 0.70)')
print('  mAP50-95 :', round(float(m.box.map), 4))
print('  Precision:', round(float(m.box.mp), 4))
print('  Recall   :', round(float(m.box.mr), 4))
print()
names = best.names
for i, ap in enumerate(m.box.ap50):
    print(f'  {names.get(i, i):12s} AP50 = {ap:.4f}')

## 7. Tải trọng số về

In [ ]:
import json, shutil

shutil.copy('/content/runs/saferoad_yolo11n/weights/best.pt',
            '/content/drive/MyDrive/saferoad_yolo11n_best.pt')

summary = {
    'mAP50': round(float(m.box.map50), 4),
    'mAP50_95': round(float(m.box.map), 4),
    'precision': round(float(m.box.mp), 4),
    'recall': round(float(m.box.mr), 4),
    'per_class_AP50': {best.names.get(i, str(i)): round(float(ap), 4)
                       for i, ap in enumerate(m.box.ap50)},
    'imgsz': 960, 'epochs_config': 60, 'freeze': 10,
}
with open('/content/drive/MyDrive/saferoad_finetune_metrics.json', 'w') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

from google.colab import files
files.download('/content/runs/saferoad_yolo11n/weights/best.pt')

## 8. Đưa trọng số mới vào hệ thống

Trên máy của bạn:

```bash
# Chép file vừa tải về vào thư mục models/
copy saferoad_yolo11n_best.pt models\saferoad_yolo11n.pt      # Windows
# cp saferoad_yolo11n_best.pt models/saferoad_yolo11n.pt       # Linux/macOS

# Chạy lại đánh giá trên dữ liệu thật với trọng số mới
python -m saferoad evaluate-real \
    --weights models/saferoad_yolo11n.pt \
    --out data/outputs/evaluation_real_finetuned.json
```

Sau đó đặt hai file `evaluation_real.json` (trước) và
`evaluation_real_finetuned.json` (sau) cạnh nhau — đó chính là bảng
**"trước / sau fine-tune"** cho mục Đánh giá trong tài liệu Bảng C.